# SAGARNETRA — 5-model SAR oil-spill bake-off (Colab T4)

Trains the five candidates under one harness, scores them on the Krestenitis 110-image test split, and exports the winner to ONNX for the API. Thin driver — all logic lives in `ml/{dataset,models,train,export}.py` so it stays in step with the repo and CI.

**Before you run:** set the runtime to **GPU (T4)** and have the Krestenitis dataset ready (the manual blocker — request from MKLab). Point `DATA_ROOT` at a folder containing `train/` and `test/`, each with an images dir and a masks dir (`labels_1D` preferred).

In [ ]:
# 1. Get the code. Either clone the repo or upload the ml/ folder.
!git clone https://github.com/Pjmahendra/SAGARNETRA.git
%cd SAGARNETRA
!pip install -q -r ml/requirements-train.txt

In [ ]:
# 2. Point at the dataset. Mount Drive, or unzip the Krestenitis archive you were granted.
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/krestenitis'   # <-- edit: must contain train/ and test/

import os
assert os.path.isdir(DATA_ROOT), f'not found: {DATA_ROOT}'
print('contents:', os.listdir(DATA_ROOT))

In [ ]:
# 3. Train all five (identical loader/loss/tiles). ~40 epochs on a T4; drop --epochs to smoke-test first.
#    Short on time? add:  --models unet_scratch unet_resnet34   (the priority pair)
!python -m ml.train --data-root "$DATA_ROOT" --epochs 40 --batch-size 8

In [ ]:
# 4. Read the comparison table and the auto-recommended winner.
import json
m = json.load(open('ml/metrics.json'))
print('recommended:', m['recommended'], '| test images:', m['test_images'])
for r in m['models']:
    print(f"{r['name']:<22} mIoU={r['miou']:.3f}  oil={r['iou'].get('oil')}  lookalike={r['iou'].get('lookalike')}  {r['params_m']}M  {r['cpu_ms']}ms")

In [ ]:
# 5. Export the winner to ONNX + sidecar. Uses metrics.json['recommended']; override --model-name to pick another.
WINNER = m['recommended']
!python -m ml.export --model-name $WINNER --weights ml/weights/$WINNER.pt

In [ ]:
# 6. Download the artefacts to commit into the API server (ml/weights/ is git-ignored except .gitkeep).
from google.colab import files
files.download(f'ml/weights/{WINNER}.onnx')
files.download(f'ml/weights/{WINNER}.json')
files.download('ml/metrics.json')
print('Place the .onnx + .json in ml/weights/ and commit metrics.json. /api/health flips to engine=unet automatically.')